# Data Acquisition & EDA

Load F-DATA + PM100, descriptive stats, EDA plots (Track B5): target distributions, correlation heatmaps, job-count-over-time, PM100 intra-job power traces.

See the conceptualization plan and EXPERIMENT_TRACKER.md for full context.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt

from src import config, features, metrics, plotting, models, roofline


## Load raw data

In [ ]:
FDATA_DIR = "../data/raw/fdata"
PM100_PATH = "../data/raw/pm100/pm100_job_table.parquet"

import glob

fdata_files = sorted(glob.glob(f"{FDATA_DIR}/*.parquet"))
print(f"F-DATA monthly files found: {len(fdata_files)} / 38 expected")
print(fdata_files)

import pandas as pd
fdata = pd.concat([pd.read_parquet(f) for f in fdata_files], ignore_index=True)
pm100 = pd.read_parquet(PM100_PATH)

# F-DATA's datetime columns load as plain strings — parse once, up front,
# rather than repeatedly downstream.
FDATA_DATETIME_COLS = ["adt", "qdt", "schedsdt", "deldt", "sdt", "edt"]
for c in FDATA_DATETIME_COLS:
    fdata[c] = pd.to_datetime(fdata[c], errors="coerce")

pm100["submit_time"] = pd.to_datetime(pm100["submit_time"], unit="s", errors="coerce")
pm100["eligible_time"] = pd.to_datetime(pm100["eligible_time"], unit="s", errors="coerce")
pm100["start_time"] = pd.to_datetime(pm100["start_time"], unit="s", errors="coerce")
pm100["end_time"] = pd.to_datetime(pm100["end_time"], unit="s", errors="coerce")

print("\nfdata shape:", fdata.shape)
print("pm100 shape:", pm100.shape)
print("\nembedding column dtype:", fdata["embedding"].dtype, "- one 384-dim ndarray per row, excluded from describe() below")


**Note:** as of this notebook's last run, only 1 of the 38 expected F-DATA monthly files is present (`fugaku_22_01.parquet`, Jan 2022). The chronological split, rolling-window drift check (Decision #18), and full-scale run (Decision #10) all need the remaining months acquired before they're meaningful — this cell just reflects whatever's actually in `data/raw/fdata/` at the time it's run.

## Schema check

In [ ]:
print("=== F-DATA dtypes ===")
print(fdata.dtypes)
print("\n=== PM100 dtypes ===")
print(pm100.dtypes)


## Confirm PM100 has no FLOP/performance-counter fields

This was already verified directly against PM100's `documentation/job_features.md` during planning (Decision #15) — re-checking here against the actual loaded dataframe as a belt-and-suspenders confirmation, not a new finding.

In [ ]:
flop_like = [c for c in pm100.columns if any(k in c.lower() for k in ["flop", "perf", "cycle", "instr", "bandwidth", "mbwidth", "opint"])]
print("PM100 columns matching FLOP/performance-counter-like names:", flop_like)
assert flop_like == [], "Unexpected: PM100 appears to have a performance-counter field after all — revisit Decision #15"

print("\nF-DATA equivalents (for contrast):", [c for c in fdata.columns if c in ("flops", "mbwidth", "opint", "pclass", "perf1", "perf2", "perf3", "perf4", "perf5", "perf6")])


## Missingness

In [ ]:
print("=== F-DATA missing value counts (nonzero only) ===")
print(fdata.isna().sum().loc[lambda s: s > 0])
print("\n=== PM100 missing value counts (nonzero only) ===")
print(pm100.isna().sum().loc[lambda s: s > 0])


## Descriptive statistics

`embedding` is excluded — it's a 384-dim `ndarray` per row, not something `describe()`'s unique/top/freq machinery can handle (this is what caused the first execution attempt to hang: pandas trying to hash/compare unhashable numpy arrays across 1.3M rows).

In [ ]:
fdata_numeric = fdata.select_dtypes(include="number")
fdata_categorical = fdata.drop(columns=["embedding"]).select_dtypes(exclude="number")

print("=== F-DATA numeric columns ===")
display(fdata_numeric.describe().T)
print("\n=== F-DATA non-numeric columns (excluding embedding) ===")
display(fdata_categorical.describe().T)


In [ ]:
pm100.describe(include="all").T  # no embedding/array columns here, so this is fine as-is


## Target distributions (raw vs. log1p)

Targets per Decision #2/#3: F-DATA → `duration` (execution time), `mmszu` (memory used), `avgpcon` (power). PM100 → `run_time`, `mem_alloc` (no "used" field exists — this is the Decision #2 fallback case, not a choice), `node_power_consumption`.

In [ ]:
import numpy as np

for name, col in features.FDATA_TARGETS.items():
    vals = fdata[col].dropna().to_numpy()
    vals = vals[vals >= 0]  # log1p requires non-negative
    fig = plotting.plot_target_distribution(vals, np.log1p(vals), f"F-DATA {name} ({col})")
    plt.show()


In [ ]:
for name, col in features.PM100_TARGETS.items():
    series = pm100[col].dropna()
    if len(series) and hasattr(series.iloc[0], "__len__"):
        # node_power_consumption is a per-job time series (20s-interval
        # samples, Decision #13's LSTM/TCN sequence input) — reduce to a
        # per-job mean for this scalar distribution plot; the raw sequence
        # is used as-is later for LSTM/TCN, not flattened here.
        vals = series.apply(lambda arr: float(np.mean(arr))).to_numpy()
        print(f"PM100 {name} ({col}): array-valued per job (e.g. length {len(series.iloc[0])} samples) — plotting per-job mean")
    else:
        vals = series.to_numpy()
    vals = vals[vals >= 0]
    fig = plotting.plot_target_distribution(vals, np.log1p(vals), f"PM100 {name} ({col})")
    plt.show()


## Job-count-over-time

Justifies the chronological train/test split, and — once more F-DATA months are present — will show whether there's an obvious basis for the rolling-window drift check window size (Decision #18).

In [ ]:
fdata.set_index("adt").resample("D").size().plot(figsize=(12, 3), title="F-DATA jobs per day")
plt.show()

pm100.set_index("submit_time").resample("D").size().plot(figsize=(12, 3), title="PM100 jobs per day")
plt.show()


## Tier A/B feature-matrix sanity check

Confirms `src/features.py`'s column lists actually apply cleanly to the loaded data (already validated once from the command line — this is the in-notebook, repeatable version of that check).

In [ ]:
for name, df, dataset in [("F-DATA", fdata, "fdata"), ("PM100", pm100, "pm100")]:
    tier_a = features.build_tier_a_features(df, dataset)
    tier_b = features.build_tier_b_features(df, dataset)
    features.assert_no_tier_leakage(list(tier_a.columns), "A", dataset)
    features.assert_no_tier_leakage(list(tier_b.columns), "B", dataset)
    print(f"{name}: Tier A {tier_a.shape}, Tier B {tier_b.shape} — OK, no leakage")
